# Figure 5: CEBRA Population Manifold Analysis

**Paper**: Parodi et al. (2026)

This notebook reproduces Figure 5, which uses CEBRA (v0.4.0) to learn nonlinear neural manifolds and assess:
- **Panel C**: Height and motif decoding from CEBRA embeddings vs PCA vs raw neural activity
- **Panel D**: Transition dynamics — neural velocity increases at behavioral state transitions
- **Panel E**: Context-dependent coding — same behavior encoded differently at different spatial locations

**CEBRA** is a self-supervised contrastive learning method that finds low-dimensional neural embeddings aligned with behavioral labels.

In [ ]:
import sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 150, 'axes.spines.top': False, 'axes.spines.right': False,
})

SESSIONS = ["M2_0526", "M2_0605", "M1_0525", "M1_0609", "M1_0613"]
FIG5_DIR = REPO_ROOT / "results" / "fig5_cebra"
print("Setup complete.")

## Load Production Results (5000 iterations, 5 sessions)

In [ ]:
# Load all session results
all_results = {}
for sess in SESSIONS:
    results_path = FIG5_DIR / sess / "results.json"
    if results_path.exists():
        with open(results_path) as f:
            all_results[sess] = json.load(f)
        print(f"  {sess}: loaded")
    else:
        print(f"  {sess}: MISSING")

print(f"\nLoaded {len(all_results)} sessions")

### Fig 5C: Decoding Comparison (CEBRA vs PCA vs Raw)

In [ ]:
# Fig 5C: Height and motif decoding comparison across sessions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ["CEBRA", "PCA-8D", "Raw"]
method_colors = ["#1976D2", "#FF9800", "#4CAF50"]

for ax, (task, key_prefix, chance_key) in zip(axes, [
    ("Height (3-class)", "height_3class", 0.333),
    ("Motif", "motif", None),
]):
    x = np.arange(len(all_results))
    width = 0.25

    for i, (method, color) in enumerate(zip(methods, method_colors)):
        if task.startswith("Height"):
            keys = [f"{key_prefix}_CEBRA-SpatialLite-8D kNN",
                    f"{key_prefix}_PCA-8D kNN",
                    f"{key_prefix}_Raw logistic"]
        else:
            keys = [f"{key_prefix}_CEBRA-Behavior-8D kNN",
                    f"{key_prefix}_PCA-8D kNN",
                    f"{key_prefix}_Raw logistic"]

        vals = [all_results[s].get(keys[i], 0) for s in all_results]
        ax.bar(x + i * width, vals, width, label=method, color=color, alpha=0.8)

    # Chance line
    if chance_key:
        ax.axhline(chance_key, color="gray", linestyle="--", linewidth=1, label=f"Chance ({chance_key:.1%})")

    ax.set_xticks(x + width)
    ax.set_xticklabels(list(all_results.keys()), rotation=30, ha="right")
    ax.set_ylabel("Balanced Accuracy")
    ax.set_title(f"{task} Decoding")
    ax.legend(fontsize=9)

fig.suptitle("Fig 5C: Decoding from Neural Manifold", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Fig 5D-E: Transition Dynamics and Context-Dependent Coding

In [ ]:
# Fig 5D-E: Transition velocity ratio + context coding ratio
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sessions_list = list(all_results.keys())
subjects = ["M2" if "M2" in s else "M1" for s in sessions_list]
subject_colors = ["#1976D2" if s == "M2" else "#D32F2F" for s in subjects]

# Panel D: Velocity ratio at transitions
ax = axes[0]
vel_ratios = [all_results[s].get("velocity_ratio", 0) for s in sessions_list]
vel_pca = [all_results[s].get("velocity_ratio_pca", 0) for s in sessions_list]

x = np.arange(len(sessions_list))
ax.bar(x - 0.15, vel_ratios, 0.3, color="#1976D2", alpha=0.8, label="CEBRA")
ax.bar(x + 0.15, vel_pca, 0.3, color="#FF9800", alpha=0.8, label="PCA")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="No change")
ax.set_xticks(x)
ax.set_xticklabels(sessions_list, rotation=30, ha="right")
ax.set_ylabel("Velocity Ratio (peak / baseline)")
ax.set_title("Fig 5D: Transition Dynamics")
ax.legend(fontsize=9)

# Panel E: Context coding ratio
ax = axes[1]
ctx_ratios = [all_results[s].get("context_ratio", 0) for s in sessions_list]
ctx_pca = [all_results[s].get("context_ratio_pca", 0) for s in sessions_list]

ax.bar(x - 0.15, ctx_ratios, 0.3, color="#1976D2", alpha=0.8, label="CEBRA")
ax.bar(x + 0.15, ctx_pca, 0.3, color="#FF9800", alpha=0.8, label="PCA")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="No context effect")
ax.set_xticks(x)
ax.set_xticklabels(sessions_list, rotation=30, ha="right")
ax.set_ylabel("Context Ratio (between / within)")
ax.set_title("Fig 5E: Context-Dependent Coding")
ax.legend(fontsize=9)

fig.suptitle("Fig 5D-E: Manifold Dynamics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary table
print("\n=== Fig 5 Summary ===")
print(f"{'Session':<10s} {'Vel CEBRA':>10s} {'Vel PCA':>10s} {'Ctx CEBRA':>10s} {'Ctx PCA':>10s}")
print("-" * 52)
for s in sessions_list:
    r = all_results[s]
    print(f"{s:<10s} {r.get('velocity_ratio',0):>10.2f} {r.get('velocity_ratio_pca',0):>10.2f} "
          f"{r.get('context_ratio',0):>10.2f} {r.get('context_ratio_pca',0):>10.2f}")